# Roman workbook 00 — data and cycle map

This workbook establishes the input boundary for the five-chamber Roman refactor. It does **not** calculate chamber radii, driver allocations, ignition, or cycle closure. Its job is to make the accepted architecture and packaged nuclear datasets easy to inspect before any new physical model is selected.

In [ ]:
from pprint import pprint

import cno_sweep
from cno_sweep import BUILTIN_DATASETS, load_roman_mainline
from cno_sweep.constants import AVOGADRO
from cno_sweep.workbook import (
    neutron_cross_section_rows,
    reaction_rows,
    reactivity_rows,
    roman_chamber_rows,
)

print(f'cno_sweep {cno_sweep.__version__}')
pprint(BUILTIN_DATASETS)

## Accepted architecture

Every row below is one distinct central blast-chamber chemistry. Trajan is recorded separately as the distributed p+N15 mantle.

In [ ]:
manifest = load_roman_mainline()
pprint(roman_chamber_rows(), sort_dicts=False)
pprint(manifest['distributed_driver'], sort_dicts=False)
assert manifest['central_chamber_count'] == 5
assert all(len(chamber['central_reaction_ids']) == 1 for chamber in manifest['chambers'])
assert manifest['chambers'][1]['excluded_from_central_fuel'] == ['h1']

## Mainline reaction ledger

The table exposes Q values, decay times, and automatic baryon/charge checks. It does not imply that the desired branch dominates competing channels.

In [ ]:
mainline_ids = [
    'c12-p-g-n13', 'n13-beta-c13',
    'c13-a-n-o16', 'n-p-g-d',
    'o16-p-g-f17', 'f17-beta-o17',
    'o17-p-a-n14',
    'n14-p-g-o15', 'o15-beta-n15',
    'n15-p-a-c12',
]
ledger = reaction_rows(mainline_ids)
pprint(ledger, sort_dicts=False)
assert all(row['baryon_conserved'] and row['charge_conserved'] for row in ledger)

## Rate comparison at T9 = 5

REACLIB reports molar rates in cm³ mol⁻¹ s⁻¹. The library evaluates particle-pair reactivity in m³/s, so the final expression converts back only for comparison with the architecture document. These are screening fits whose reaction-specific validity ranges still require review.

In [ ]:
temperature_t9_5_keV = 5.0e9 / 1.160451812e7
rate_ids = [
    'c13-a-n-o16',
    'o16-p-g-f17',
    'o17-p-a-n14',
    'n14-p-g-o15',
    'n15-p-a-c12',
]
rates = reactivity_rows(rate_ids, [temperature_t9_5_keV])
for row in rates:
    row['na_reactivity_cm3_mol_s'] = row['reactivity_m3_s'] * AVOGADRO * 1.0e6
pprint(rates, sort_dicts=False)

### Refactor-boundary discrepancy

With the pinned 2026-06-09 REACLIB subset, C13(alpha,n) evaluates to approximately **3.50e6 cm³ mol⁻¹ s⁻¹ at T9=5**, while `ROMAN_REFACTOR.md` quotes approximately **8.62e5**. O16(p,gamma) and N15(p,alpha) reproduce the document's quoted values. The Constantine rate discrepancy must be traced before workbook 10 uses either number; this workbook intentionally displays the evaluated source value rather than silently forcing agreement.

## Constantine neutron-data inventory

This is only a cross-section lookup. Workbook 10 must add source spectrum, time-dependent composition, scattering, inverse reactions, geometry, expansion, and a recoverable-D criterion.

In [ ]:
constantine_xs = neutron_cross_section_rows(
    ['h1', 'he4', 'c13', 'o16', 'n14', 'n15'],
    [2.0, 14.1],
)
pprint(constantine_xs, sort_dicts=False)

## Boundary carried forward

The next workbook must not assume that a neutron which deposits energy becomes D, or that D created in hot fuel survives. Constantine succeeds only if its neutron exits the proton-free hot source, is captured on physically separate ordinary hydrogen, and leaves recoverable D after the event.